In [ ]:
from IPython.display import display, HTML

display(HTML(
    """
    <head> 
        <style>
            p {text-align: left; font-family: Times New Roman; font-size: 12pt;}
            h1 {font-family: Times New Roman; font-size: 50pt; text-align: center}
        </style>
        <body>
            <h1>PROYECTO:<br>CURVA DE OFERTA AGREGADA</h1>
            <h3 style = "font-family: Times New Roman; text-align: center;">Juan Esteban Roa<br>Danna Alejandra Salguero</h3>
            <p style = "background-color: tomato">Este proyecto trata sobre la obtención del precio de bolsa a partir de la demanda pronosticada y la curva de oferta agregada, identificando agentes y tecnología que ponen el precio marginal.</p>
            <h2 style = "text-align:left; font-family:Times New Roman;">Conceptos:</h2>
            <p><b>Oferta agregada (mérito de despacho)</b>:Es la construcción horaria donde se ordenan todas las ofertas de energía de menor a mayor precio. El punto exacto en el que esta curva escalonada corta la demanda real de esa hora determina el Precio de Bolsa.</p>
            <p><b>Disponibilidad real</b>: La cantidad máxima de megavatios (MW) que la planta puede generar físicamente (limitada por agua en embalses, fallas mecánicas o combustible disponible).</p>
            <p><b>Precio de oferta</b>: El valor (en COP/kWh) al que está dispuesta a vender esa energía.</p>

            <h2 style = "text-align:left; font-family:Times New Roman;">Estrategia:</h2>
            <ol style = "font-family:Times New Roman">
                <li>Obtención de datos por medio de la API de XM.</li>
                <li>Delimitación del periodo</li>
                <li>Organización y tratamiento de los Dataframes</li>
                <li>Graficación de los datos</li>
            </ol>

        </body>
    </head>
    """
))

In [ ]:
display(HTML(
    """
    <head>
        <style>
            li {font-family: Times New Roman;}
            h1 {font-size: 30pt; font-family: Times New Roman;text-align: left}
        </style>
    </head>
    <body>
        <h1>1. Obtención de datos</h1>
        <h2>Fecha: 1 Octubre 2025 - 1 Diciembre 2025</h2>
        <ul> Se usarán las variables de:
            <li>Precio ofertado por los generadores</li>
            <li>Disponibilidad declarada por los generadores</li>
            <li>Precio de bolsa nacional</li>
            <li>Generación ideal</li>    
            <li>Proyección de demanda de UPME en escenario bajo</li>
        </ul>
        <h2 style = "text-align:left;">Detalles de las variables</h2>
        <img src = "disponibilidad.png"></img>
        <img src = "oferta.png"></img>
        <img src = "operadores.png"></img>
        <img src = "bolsa.png"></img>
        <h2 style = "color:#EDC721;">Demanda de Energia Proyectada UPME Escenario Bajo por Sistema:</h2>
        <p>Id = EscDemUPMEBajo. Type = MonthlyEntities. Units = kWh  </p>
    </body>
    """
))

In [ ]:
FECHA_INICIO = "2025-10-01"
FECHA_FIN = "2025-12-01"
METRICAS = {
    "oferta_precio": "PrecOferDesp",     # Precio de Oferta del Despacho   (Recurso)
    "disponibilidad": "DispoDeclarada",   # Disponibilidad Declarada        (Recurso)
    "gen_ideal": "GeneIdea",             # Generación Ideal por Recurso    (Recurso)
    "mpo": "MaxPrecOferNal",             # Máximo Precio de Oferta         (Sistema)
    "pbn": "PrecBolsNaci",               # Precio de Bolsa Nacional        (Sistema)
    "listRecursos":"ListadoRecursos",    #Listado de Recursos
    "listAgentes":"ListadoAgentes"       #Listado de Agentes

}

entidad = ["Recurso","Recurso","Recurso","Sistema","Sistema","Sistema","Sistema"]

api = ReadDB()

df_variables = {}
i = 0
for name, code in METRICAS.items():
    var = api.request_data(code,entidad[i],FECHA_INICIO,FECHA_FIN)
    df_variables[name] = var
    i+=1

In [ ]:
agentes_nombres = (
    df_variables["listRecursos"].drop_duplicates(subset=['Values_Code'])
      .set_index('Values_Code')['Values_Name']
      .to_dict()
)

agentes_actividad = (df_variables["listRecursos"].drop_duplicates(subset = ['Values_Name']).set_index('Values_Name')['Values_Type'].to_dict())

In [ ]:
#nuevos_nombres_oferta = [f"oferta_hora_{x}" for x in range(1,25)]

df_oferta = df_variables["oferta_precio"]
df_oferta.rename(columns = {df_oferta.columns.values[1+i]:f"oferta_hora_{i}" for i in range(1,25) }, inplace = True)

#nuevos_nombres_dip = [f"disp_hora_{x}" for x in range(1,25)]

df_disponibilidad = df_variables["disponibilidad"]
df_disponibilidad.rename(columns = {df_disponibilidad.columns.values[1+i]:f"disp_hora_{i}" for i in range(1,25) }, inplace = True)


df_oferta['Date'] = pd.to_datetime(df_oferta['Date'])
df_disponibilidad['Date'] = pd.to_datetime(df_disponibilidad['Date'])

print(df_disponibilidad.head())

        Id Values_code  disp_hora_1  disp_hora_2  disp_hora_3  disp_hora_4  \
0  Recurso        2QBW        18420        18420        18420        18420   
1  Recurso        2QEK        35000        35000        35000        35000   
2  Recurso        2QRL          650          650          650          650   
3  Recurso        2QV2         2000         2000         2000         2000   
4  Recurso        2R22        18000        18000        18000        18000   

   disp_hora_5  disp_hora_6  disp_hora_7  disp_hora_8  ...  disp_hora_16  \
0        18420        18420        18420        18420  ...         18420   
1        35000        35000        35000        35000  ...         35000   
2          650          650          650          650  ...           650   
3         2000         2000         2000            0  ...          2000   
4        18000        18000        18000        18000  ...         18000   

   disp_hora_17  disp_hora_18  disp_hora_19  disp_hora_20  disp_hora_21  \

In [ ]:
# 2. Limpiar espacios en blanco y asegurar que sean texto en los códigos
df_oferta['Values_code'] = df_oferta['Values_code'].astype(str).str.strip()
df_disponibilidad['Values_code'] = df_disponibilidad['Values_code'].astype(str).str.strip()

df_oferta['Id'] = df_oferta['Id'].astype(str).str.strip()
df_disponibilidad['Id'] = df_disponibilidad['Id'].astype(str).str.strip()

# 3. Intentar el merge nuevamente
df_oferta_disp = pd.merge(
    left=df_disponibilidad, 
    right=df_oferta, 
    on=['Id','Date','Values_code'], 
    how='inner'
)

print("Filas después del merge:", len(df_oferta_disp))

Filas después del merge: 12289


In [ ]:
df_oferta_disp["Nombre_Planta"] = df_oferta["Values_code"].astype(str).str.strip().map(agentes_nombres)
print(df_oferta_disp)

# Insertar actividad

df_oferta_disp["Tecnologia"] = df_variables["listRecursos"]["Values_Name"].astype(str).str.strip().map(agentes_actividad)

            Id Values_code  disp_hora_1  disp_hora_2  disp_hora_3  \
0      Recurso        2QEK        35000        35000        35000   
1      Recurso        3DDT            0            0            0   
2      Recurso        3ENA       200000       200000       200000   
3      Recurso        3HF5            0            0            0   
4      Recurso        3INX            0            0            0   
...        ...         ...          ...          ...          ...   
12284  Recurso        URA1       338000       338000       338000   
12285  Recurso        ZPA2        34000        34000        34000   
12286  Recurso        ZPA3        63000        63000        63000   
12287  Recurso        ZPA4            0            0            0   
12288  Recurso        ZPA5        63000        63000        63000   

       disp_hora_4  disp_hora_5  disp_hora_6  disp_hora_7  disp_hora_8  ...  \
0            35000        35000        35000        35000        35000  ...   
1            

In [ ]:
display(HTML(
    """
    <h1>Para la construcción de la curva de oferta agregada</h1>
    <img src = "diapositiva_oferta.png">Definición de D y P para la oferta de generación.</img>
    <img src = "diapositiva3.png">Mérito de despacho</img>
    """
))

In [ ]:
# 1. Identificar las 24 columnas de horas que quieres apilar
horas_cols = [f"oferta_hora_{x}" for x in range(1, 25)]

# 2. Identificar automáticamente todas las demás columnas (Id, Values_code, Date, Tecnología, etc.)
# Esto evita que tengas que escribirlas una por una.
otras_columnas = [col for col in df_oferta.columns if col not in horas_cols]
horas_cols_disp = [f"disp_hora_{x}" for x in range(1, 25)]
otras_columnas_disp = [col for col in df_disponibilidad.columns if col not in horas_cols_disp]
# 3. Aplicar melt para pasar las 24 columnas a filas
df_largo = df_oferta.melt(
    id_vars=otras_columnas,        # Columnas que se mantendrán fijas y cuyos valores se repetirán
    value_vars=horas_cols,         # Las 24 columnas horarias que queremos "apilar"
    var_name='Hora_Original',      # Nombre temporal para la columna que indicará la hora (ej. 'oferta_hora_1')
    value_name='Precio_Oferta'     # El valor numérico del precio para esa hora
)

df_largo_disp = df_disponibilidad.melt(
    id_vars=otras_columnas_disp,        # Columnas que se mantendrán fijas y cuyos valores se repetirán
    value_vars=horas_cols_disp,         # Las 24 columnas horarias que queremos "apilar"
    var_name='Hora_Original',      # Nombre temporal para la columna que indicará la hora (ej. 'oferta_hora_1')
    value_name='disponibilidad'     # El valor numérico del precio para esa hora
)

df_largo['Hora_Num'] = df_largo['Hora_Original'].str.extract(r'(\d+)').astype(int)
df_largo_disp['Hora_Num'] = df_largo_disp['Hora_Original'].str.extract(r'(\d+)').astype(int)

df_largo['Date'] = pd.to_datetime(df_largo['Date']) + pd.to_timedelta(df_largo['Hora_Num'] - 1, unit='h')
df_largo_disp['Date'] = pd.to_datetime(df_largo_disp['Date']) + pd.to_timedelta(df_largo_disp['Hora_Num'] - 1, unit='h')
df_largo = df_largo.drop(columns=['Hora_Original', 'Hora_Num'])
df_largo_disp = df_largo_disp.drop(columns=['Hora_Original', 'Hora_Num'])

print(df_largo_disp.head())

        Id Values_code       Date  disponibilidad
0  Recurso        2QBW 2025-10-01           18420
1  Recurso        2QEK 2025-10-01           35000
2  Recurso        2QRL 2025-10-01             650
3  Recurso        2QV2 2025-10-01            2000
4  Recurso        2R22 2025-10-01           18000


In [ ]:
df_largo_oferta_disp = pd.merge(left = df_largo, right = df_largo_disp, on = ["Id","Values_code","Date"], how = "inner")
df_largo_oferta_disp["Nombre_Planta"] = df_oferta["Values_code"].astype(str).str.strip().map(agentes_nombres)
df_largo_oferta_disp["Tecnologia"] = df_variables["listRecursos"]["Values_Name"].astype(str).str.strip().map(agentes_actividad)

print(df_largo_oferta_disp.head())

        Id Values_code       Date  Precio_Oferta  disponibilidad  \
0  Recurso        2QEK 2025-10-01         97.815           35000   
1  Recurso        3DDT 2025-10-01         88.362               0   
2  Recurso        3ENA 2025-10-01        814.351          200000   
3  Recurso        3HF5 2025-10-01         88.365               0   
4  Recurso        3INX 2025-10-01         88.000               0   

         Nombre_Planta  Tecnologia  
0             SALTO II  HIDRAULICA  
1  LATAM SOLAR LA LOMA  HIDRAULICA  
2             TESORITO  HIDRAULICA  
3            FUNDACION  HIDRAULICA  
4           CARACOLI I  HIDRAULICA  


In [ ]:
# Usamos .pivot() indicando todas las columnas que hacen única a cada fila
df_reorganizado = df_largo_oferta_disp.pivot_table(
    index=['Id', 'Values_code', 'Date', 'Precio_Oferta', 'Nombre_Planta'],
    columns='Tecnologia',
    values='disponibilidad',
    aggfunc='first'  # Toma el primer valor disponible sin modificar los números
).reset_index()

print(df_reorganizado.head())

Tecnologia       Id Values_code       Date  Precio_Oferta Nombre_Planta  \
0           Recurso        2QEK 2025-10-01         97.815      SALTO II   
1           Recurso        2QEK 2025-10-02         97.515      SALTO II   
2           Recurso        2QEK 2025-10-03         97.815      SALTO II   
3           Recurso        2QEK 2025-10-04         97.815      SALTO II   
4           Recurso        2QEK 2025-10-05         97.815      SALTO II   

Tecnologia  COGENERADOR   EOLICA  HIDRAULICA    SOLAR  Solar  TERMICA  
0                   NaN      NaN     35000.0      NaN    NaN      NaN  
1                   NaN      NaN         NaN  35000.0    NaN      NaN  
2                   NaN  35000.0         NaN      NaN    NaN      NaN  
3                   NaN      NaN         NaN  35000.0    NaN      NaN  
4                   NaN      NaN         NaN  35000.0    NaN      NaN  


In [ ]:
display(HTML(
    """
    <h1 style = "font-family: Times New Roman;">Búsqueda de variable de demanda pronósticada</h1>
    """
))

In [ ]:
df_metricas = api.get_collections()

# Puedes filtrar para buscar las palabras clave como "Demanda" o "Dema"
demandas_disponibles = df_metricas[df_metricas['MetricName'].str.contains('Demanda|Dema', case=False, na=False)]
print(demandas_disponibles)

             MetricId                                         MetricName  \
0            DemaReal                           Demanda Real por Sistema   
1            DemaReal                            Demanda Real por Agente   
3            DemaCome                      Demanda Comercial por Sistema   
12           DemaCome                       Demanda Comercial por Agente   
33             DemaOR                          Demanda por OR por Agente   
51        DemaComeReg      Demanda Comercial Mercado Regulado por Agente   
52      DemaComeNoReg   Demanda Comercial Mercado No Regulado por Agente   
53        DemaComeReg     Demanda Comercial Mercado Regulado por Sistema   
54      DemaComeNoReg  Demanda Comercial Mercado No Regulado por Sistema   
56      DemaComeNoReg                         Demanda Comercial por CIIU   
84           DemaCome      Demanda Comercial por MercadoComercializacion   
103       DemaRealReg                       Demanda Real Regulada Agente   
104       De

In [ ]:
api.get_collections("EscDemUPMEBajo")

,MetricId,MetricName,Entity,MaxDays,Type,Url,Filter,MetricUnits,MetricDescription
184,EscDemUPMEBajo,Demanda de Energia Proyectada UPME Escenario B...,Sistema,731,MonthlyEntities,https://servapibi.xm.com.co/monthly,No aplica,kWh,proyecciones de demanda de energía realizada p...


In [ ]:
display(HTML(
"""
<h1 style = "font-family: Times New Roman;">Gráfica</h1>
<p>Para el día 1 de octubre de 2025</p>
"""
))

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_curva['Centro_X'],
    y=df_curva['Precio_Oferta'],
    width=df_curva['disponibilidad'], # El ancho del bloque representa los MW de disponibilidad
    customdata=df_curva[['Nombre_Planta', 'Tecnologia', 'Values_code', 'disponibilidad']],
    hovertemplate=(
        "<b>Planta:</b> %{customdata[0]}<br>"
        "<b>Tecnología:</b> %{customdata[1]}<br>"
        "<b>Código:</b> %{customdata[2]}<br>"
        "<b>Precio de Oferta:</b> %{y:,.2f} COP/kWh<br>"
        "<b>Disponibilidad:</b> %{customdata[3]:,.2f} kWh<br>"
        "<extra></extra>"
    ),
    marker=dict(
        color=pd.factorize(df_curva['Tecnologia'])[0], # Colores automáticos por tipo de tecnología
        colorscale='Viridis',
        line=dict(color='black', width=0.5) # Línea sutil para separar visualmente los bloques
    )
))

# 8. Ajustes de diseño de la gráfica
fig.update_layout(
    title=f'<b>Curva de Mérito de Despacho </b><br><sup>Fecha: {fecha_objetivo}</sup>',
    xaxis_title='<b>Disponibilidad Acumulada del Sistema (kWh)</b>',
    yaxis_title='<b>Precio de Oferta (COP/kWh)</b>',
    template='plotly_white',
    hovermode='x',
    bargap=0 # Evita espacios vacíos artificiales entre los bloques contiguos
)

fig.show()

In [ ]:
df_demanda_UPME_Bajo = api.request_data("EscDemUPMEBajo","Sistema",FECHA_INICIO,FECHA_FIN)
print(df_demanda_UPME_Bajo)

        Id         Value       Date
0  Sistema  6.864000e+09 2025-10-01
1  Sistema  6.707000e+09 2025-11-01
2  Sistema  6.707000e+09 2025-11-01
3  Sistema  6.877000e+09 2025-12-01


In [ ]:
df_variables["pbn"]

,Id,Values_code,Values_Hour01,Values_Hour02,Values_Hour03,Values_Hour04,Values_Hour05,Values_Hour06,Values_Hour07,Values_Hour08,...,Values_Hour16,Values_Hour17,Values_Hour18,Values_Hour19,Values_Hour20,Values_Hour21,Values_Hour22,Values_Hour23,Values_Hour24,Date
0,Sistema,Sistema,108.52143,108.52143,107.55943,108.52143,235.81943,258.62143,258.62143,258.62143,...,258.62143,293.62143,458.62143,807.62143,822.97243,807.62143,458.62143,293.62143,258.62143,2025-10-01
1,Sistema,Sistema,107.89475,107.89475,107.89475,107.89475,107.89475,234.61675,234.61675,234.61675,...,234.61675,258.61675,459.61675,823.96775,823.96775,708.61675,459.61675,294.61575,234.61675,2025-10-02
2,Sistema,Sistema,233.23330,233.23330,233.23330,233.23330,233.23330,233.43130,276.23330,268.39730,...,291.23330,291.23330,291.23330,605.23330,705.23330,456.23330,291.23330,276.23330,233.23330,2025-10-03
3,Sistema,Sistema,181.05610,181.05610,181.05610,181.05610,181.05610,181.05610,181.05610,181.05610,...,225.05610,255.05610,266.05610,456.05610,276.05610,276.05610,266.05610,233.25410,233.25410,2025-10-04
4,Sistema,Sistema,224.31157,224.31157,224.31157,224.31157,224.31157,224.31157,224.31157,107.58957,...,224.31157,224.31157,229.31157,279.31157,279.31157,279.31157,279.31157,229.31157,224.31157,2025-10-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,Sistema,Sistema,182.83419,182.83419,182.83419,182.83419,182.83419,182.83419,182.83419,182.83419,...,210.93419,210.93419,270.83419,258.82919,723.47519,210.93419,210.93419,210.93419,205.93419,2025-11-27
88,Sistema,Sistema,248.86655,248.86655,248.86655,248.86655,248.86655,248.86655,248.86655,248.86655,...,268.86655,268.86655,268.86655,268.86655,268.86655,268.86655,268.86655,248.86655,248.86655,2025-11-28
89,Sistema,Sistema,238.86220,238.86220,238.86220,238.86220,238.86220,238.86220,238.86220,238.86220,...,238.86220,256.86220,256.86220,287.86220,287.86220,287.86220,256.86220,238.86220,238.86220,2025-11-29
90,Sistema,Sistema,258.47069,236.47069,236.47069,236.47069,236.47069,236.47069,236.47069,236.47069,...,289.47069,289.47069,294.47069,294.47069,294.47069,294.47069,294.47069,294.47069,294.47069,2025-11-30


In [ ]:
df_variables["pbn"][df_variables["pbn"]["Date"] == fecha_objetivo]

,Id,Values_code,Values_Hour01,Values_Hour02,Values_Hour03,Values_Hour04,Values_Hour05,Values_Hour06,Values_Hour07,Values_Hour08,...,Values_Hour16,Values_Hour17,Values_Hour18,Values_Hour19,Values_Hour20,Values_Hour21,Values_Hour22,Values_Hour23,Values_Hour24,Date
0,Sistema,Sistema,108.52143,108.52143,107.55943,108.52143,235.81943,258.62143,258.62143,258.62143,...,258.62143,293.62143,458.62143,807.62143,822.97243,807.62143,458.62143,293.62143,258.62143,2025-10-01


In [ ]:
print(df_variables["demanda_comercial"])

         Id Values_code  Values_Hour01  Values_Hour02  Values_Hour03  \
0   Sistema     Sistema     8746321.55     8462238.34     8239549.80   
1   Sistema     Sistema     8821124.24     8485391.27     8280745.99   
2   Sistema     Sistema     8874272.64     8579567.22     8352791.01   
3   Sistema     Sistema     8825221.33     8501976.07     8252234.71   
4   Sistema     Sistema     8491511.33     8195892.80     7947758.69   
..      ...         ...            ...            ...            ...   
87  Sistema     Sistema     9360895.60     9039879.75     8809377.26   
88  Sistema     Sistema     9236285.81     8942361.88     8700365.45   
89  Sistema     Sistema     8695461.47     8421232.08     8194491.57   
90  Sistema     Sistema     8515097.02     8201448.82     7979510.47   
91  Sistema     Sistema     8431830.55     8170894.84     7955314.14   

    Values_Hour04  Values_Hour05  Values_Hour06  Values_Hour07  Values_Hour08  \
0      8152977.30     8308242.69     8741525.54     88

In [ ]:
crear_curva_merito(df_reorganizado,df_variables,'2025-10-01',1)

In [ ]:
crear_grid_curvas_merito(df_reorganizado, df_variables, hora=1)

In [ ]:
crear_mapa_calor_merito(df_reorganizado, df_variables, 1)